# Extract CellViT nucleus features

Preflight runs first: one real batch through the exact checkpoint, postprocessor,
DINO crop encoder and cache writer, plus a runtime estimate. Full extraction
starts only if that passes, so a bad path or scale does not burn a session.

Needs `requirements-cellvit.txt` (numba / cv2 / scikit-image). No other notebook
does.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cellvit.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
# ---- EDIT ONLY THIS CELL ----
DATASET = "pathmnist"          # pathmnist | skintissue; HistoSet needs per-source MPP
SEED = 42
DATA_PATH = DATA_PATHS[DATASET]

CHECKPOINT_CANDIDATES = [
    DATA_ROOT / "CellViT-256-x40-AMP.pth",
    Path("/kaggle/input/cellvit-checkpoints/CellViT-256-x40-AMP.pth"),
]
CHECKPOINT_PATH = str(
    next((p for p in CHECKPOINT_CANDIDATES if p.is_file()), CHECKPOINT_CANDIDATES[0])
)

# PathMNIST source pixels are 0.5 MPP. MODEL_MPP/MAGNIFICATION must match the checkpoint.
INPUT_MPP = 0.5
MODEL_MPP = 0.25
MAGNIFICATION = 40

CACHE_DIR = "/kaggle/working/cellvit_features"
DINO_MODEL = "facebook/dinov2-base"   # or a mounted local model directory
BATCH_SIZE = 2
DINO_CROP_BATCH_SIZE = 32
SMOKE_SAMPLES = 8                     # spread across the train set for the estimate
MAX_ESTIMATED_HOURS = 10.0            # fail before wasting a Kaggle session
MAX_CELLS_PER_PATCH = 16              # T4-safe; keep identical across every variant
OVERWRITE = False

assert Path(DATA_PATH).exists(), DATA_PATH
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
assert MAGNIFICATION in (20, 40)

In [ ]:
# Exact one-batch integration test. Do not continue if this cell fails.
preflight = [
    sys.executable, "scripts/preflight_cellvit.py",
    "--dataset", DATASET, "--data_path", DATA_PATH,
    "--checkpoint", CHECKPOINT_PATH, "--cache_dir", CACHE_DIR,
    "--input_mpp", str(INPUT_MPP), "--model_mpp", str(MODEL_MPP),
    "--magnification", str(MAGNIFICATION),
    "--vit_name", DINO_MODEL, "--seed", str(SEED),
    "--smoke_samples", str(SMOKE_SAMPLES),
    "--cellvit_batch_size", str(BATCH_SIZE),
    "--dino_crop_batch_size", str(DINO_CROP_BATCH_SIZE),
    "--max_estimated_hours", str(MAX_ESTIMATED_HOURS),
]
if MAX_CELLS_PER_PATCH is not None:
    preflight += ["--max_cells_per_patch", str(MAX_CELLS_PER_PATCH)]
subprocess.check_call(preflight)

In [ ]:
manifest = Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
command = [
    sys.executable, "scripts/extract_cellvit_features.py",
    "--dataset", DATASET, "--data_path", DATA_PATH,
    "--checkpoint", CHECKPOINT_PATH, "--cache_dir", CACHE_DIR,
    "--input_mpp", str(INPUT_MPP), "--model_mpp", str(MODEL_MPP),
    "--magnification", str(MAGNIFICATION),
    "--vit_name", DINO_MODEL, "--seed", str(SEED), "--device", "cuda",
    "--batch_size", str(BATCH_SIZE),
    "--dino_crop_batch_size", str(DINO_CROP_BATCH_SIZE),
]
if MAX_CELLS_PER_PATCH is not None:
    command += ["--max_cells_per_patch", str(MAX_CELLS_PER_PATCH)]
if OVERWRITE:
    command.append("--overwrite")

if manifest.is_file() and not OVERWRITE:
    print("completed cache already exists; skipping extraction:", manifest)
else:
    subprocess.check_call(command)

In [ ]:
# Zip everything this notebook produced so it downloads as one file.
import shutil

SOURCE = Path('/kaggle/working/cellvit_features')
ARCHIVE = Path("/kaggle/working/cellvit_features")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6
print(f"{ARCHIVE}.zip  ({size_mb:.1f} MB)")